# Cheminformatics: RDKit and molecular representations for the project notebooks

Role of this notebook: I am your cheminformatics tutor. We assume you learned the theory in `AdvancedChemistry`. Here we focus on tools and methods: RDKit, SMILES, descriptors, fingerprints, scaffold splits, graph conversion, fragments, conformers, and ML-ready featurization.

Primary resources:

- [RDKit Getting Started in Python](https://www.rdkit.org/docs/GettingStartedInPython.html)
- [RDKit documentation index](https://www.rdkit.org/docs/index.html)
- [TeachOpenCADD](https://projects.volkamerlab.org/teachopencadd/): excellent open cheminformatics/drug-discovery tutorials.
- [Molecular descriptors overview](https://en.wikipedia.org/wiki/Molecular_descriptor): quick taxonomy; for production work, prefer RDKit docs and primary literature.

Connection to project notebooks:

- MPNNs/DeepDDS/HiGNN need molecular graphs.
- DeepAffinity needs compound encodings and protein sequence encodings.
- GraphVAE/MolGAN/JT-VAE need validity, graph tensors, fragments, and generation metrics.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from rdkit import Chem, DataStructs
    from rdkit.Chem import AllChem, BRICS, Descriptors, Draw, QED, rdMolDescriptors, rdFingerprintGenerator
    from rdkit.Chem.Scaffolds import MurckoScaffold
    HAS_RDKIT = True
except Exception as exc:
    HAS_RDKIT = False
    RDKIT_IMPORT_ERROR = repr(exc)

try:
    import torch
    HAS_TORCH = True
except Exception as exc:
    HAS_TORCH = False
    TORCH_IMPORT_ERROR = repr(exc)

try:
    from torch_geometric.data import Data
    HAS_PYG = True
except Exception as exc:
    HAS_PYG = False
    PYG_IMPORT_ERROR = repr(exc)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Learning" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "Learning"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("RDKit:", HAS_RDKIT)
print("Torch:", HAS_TORCH)
print("PyG:", HAS_PYG)

## Lesson 1 — SMILES in, molecule object out

RDKit's central object is `Mol`. A SMILES string is a compact text representation; a `Mol` is a parsed chemical graph with atoms, bonds, valence checks, aromaticity perception, stereochemistry, and properties.

Key habit: always check whether parsing returned `None`.

In [ ]:
if not HAS_RDKIT:
    raise RuntimeError(RDKIT_IMPORT_ERROR)

smiles = ["CCO", "c1ccccc1", "CO(C)C", "C[C@H](N)C(=O)O"]
for s in smiles:
    mol = Chem.MolFromSmiles(s)
    print(f"{s:20s}", "OK" if mol is not None else "FAILED")

valid = [Chem.MolFromSmiles(s) for s in smiles if Chem.MolFromSmiles(s) is not None]
display(Draw.MolsToGridImage(valid, legends=[Chem.MolToSmiles(m) for m in valid], subImgSize=(220, 180)))

## Lesson 2 — Canonicalization, sanitization, and why the same molecule has many strings

The same molecule can be written many ways. Canonical SMILES helps deduplicate. Isomeric SMILES preserves stereochemistry when present.

In [ ]:
equiv = ["C1=CC=CN=C1", "c1cccnc1", "n1ccccc1"]
for s in equiv:
    m = Chem.MolFromSmiles(s)
    print(s, "->", Chem.MolToSmiles(m), "| isomeric:", Chem.MolToSmiles(m, isomericSmiles=True))

## Lesson 3 — Inspecting atoms and bonds

These are the raw ingredients for graph neural network features.

In [ ]:
m = Chem.MolFromSmiles("CC(=O)Nc1ccc(O)cc1")
atom_rows = []
for a in m.GetAtoms():
    atom_rows.append({
        "idx": a.GetIdx(),
        "symbol": a.GetSymbol(),
        "atomic_num": a.GetAtomicNum(),
        "degree": a.GetDegree(),
        "formal_charge": a.GetFormalCharge(),
        "hybridization": str(a.GetHybridization()),
        "aromatic": a.GetIsAromatic(),
        "in_ring": a.IsInRing(),
    })
bond_rows = []
for b in m.GetBonds():
    bond_rows.append({
        "begin": b.GetBeginAtomIdx(),
        "end": b.GetEndAtomIdx(),
        "type": str(b.GetBondType()),
        "conjugated": b.GetIsConjugated(),
        "aromatic": b.GetIsAromatic(),
        "in_ring": b.IsInRing(),
    })
display(pd.DataFrame(atom_rows))
display(pd.DataFrame(bond_rows).head())
display(Draw.MolToImage(m, size=(450, 250)))

## Lesson 4 — Descriptors: human-designed molecular numbers

Descriptors are useful baselines and sanity checks. Deep models may learn better features, but descriptors reveal whether the data distribution makes chemical sense.

In [ ]:
library = {
    "ethanol": "CCO",
    "benzene": "c1ccccc1",
    "aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "caffeine": "Cn1cnc2n(C)c(=O)n(C)c(=O)c12",
    "imatinib_core": "Cc1ccc(NC(=O)c2ccc(CN3CCNCC3)cc2)cc1",
}
rows = []
for name, smi in library.items():
    mol = Chem.MolFromSmiles(smi)
    rows.append({
        "name": name,
        "canonical_smiles": Chem.MolToSmiles(mol),
        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        "QED": QED.qed(mol),
    })
df = pd.DataFrame(rows)
display(df.round(3))
df.to_csv(DATA_DIR / "toy_molecule_descriptors.csv", index=False)
print("Saved:", DATA_DIR / "toy_molecule_descriptors.csv")

## Lesson 5 — Fingerprints and similarity

Morgan fingerprints, RDKit's implementation of circular fingerprints, are the workhorse representation behind many QSAR baselines.

Tanimoto similarity for bit vectors:

$$
T(A,B) = \frac{|A \cap B|}{|A \cup B|}.
$$

This is also a useful way to diagnose train/test leakage: if your test molecules are near-duplicates of train molecules, random-split performance can look heroic for boring reasons.

In [ ]:
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
mols = [Chem.MolFromSmiles(s) for s in library.values()]
fps = [fpgen.GetFingerprint(m) for m in mols]
names = list(library)
sim = np.zeros((len(fps), len(fps)))
for i, fpi in enumerate(fps):
    for j, fpj in enumerate(fps):
        sim[i, j] = DataStructs.TanimotoSimilarity(fpi, fpj)
plt.imshow(sim, vmin=0, vmax=1, cmap="viridis")
plt.xticks(range(len(names)), names, rotation=45, ha="right")
plt.yticks(range(len(names)), names)
plt.colorbar(label="Tanimoto")
plt.title("Morgan fingerprint similarity")
plt.tight_layout()
plt.show()

## Lesson 6 — Scaffold splitting

Random splits often overestimate molecular ML performance. Bemis--Murcko scaffolds group molecules by core ring/linker topology. Scaffold splits ask a harder question: can the model generalize to new chemical cores?

In [ ]:
scaffold_rows = []
for name, smi in library.items():
    mol = Chem.MolFromSmiles(smi)
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    scaffold_rows.append({"name": name, "smiles": smi, "scaffold": scaffold})
display(pd.DataFrame(scaffold_rows))
scaffold_mols = [Chem.MolFromSmiles(r["scaffold"]) if r["scaffold"] else Chem.MolFromSmiles(r["smiles"]) for r in scaffold_rows]
display(Draw.MolsToGridImage(scaffold_mols, legends=[r["name"] for r in scaffold_rows], subImgSize=(220, 180)))

## Lesson 7 — Conformers and 3D coordinates

RDKit can generate approximate 3D conformers with distance geometry and optimize them with molecular mechanics. This is not quantum chemistry, but it gives useful geometry for descriptors and visualization.

In [ ]:
mol = Chem.AddHs(Chem.MolFromSmiles(library["aspirin"]))
params = AllChem.ETKDGv3()
params.randomSeed = 42
ids = AllChem.EmbedMultipleConfs(mol, numConfs=10, params=params)
energies = []
for cid in ids:
    ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
    ff.Minimize(maxIts=200)
    energies.append(ff.CalcEnergy())
plt.bar(range(len(energies)), energies)
plt.xlabel("conformer id")
plt.ylabel("UFF energy / arbitrary units")
plt.title("Conformer energy spread")
plt.show()
print("Lowest energy conformer:", int(np.argmin(energies)), "energy:", min(energies))

## Lesson 8 — Molecule to PyTorch Geometric graph

This is the bridge to MPNNs, DeepDDS, HiGNN, GraphVAE-style encoders, and many graph predictors.

Below is a deliberately simple featurizer. Production notebooks usually use richer categorical encodings.

In [ ]:
def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing()),
    ]

bond_type_to_float = {
    Chem.BondType.SINGLE: 1.0,
    Chem.BondType.DOUBLE: 2.0,
    Chem.BondType.TRIPLE: 3.0,
    Chem.BondType.AROMATIC: 1.5,
}

def mol_to_pyg(mol):
    if not (HAS_TORCH and HAS_PYG):
        raise RuntimeError("Requires torch and torch_geometric")
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_pairs = []
    edge_attr = []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        feat = [bond_type_to_float.get(b.GetBondType(), 0.0), int(b.GetIsConjugated()), int(b.IsInRing())]
        edge_pairs.extend([(i, j), (j, i)])
        edge_attr.extend([feat, feat])
    edge_index = torch.tensor(edge_pairs, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, smiles=Chem.MolToSmiles(mol))

if HAS_TORCH and HAS_PYG:
    data = mol_to_pyg(Chem.MolFromSmiles(library["aspirin"]))
    print(data)
    print("x:\n", data.x[:5])
    print("edge_index shape:", data.edge_index.shape)
else:
    print("Torch/PyG missing:", globals().get("TORCH_IMPORT_ERROR"), globals().get("PYG_IMPORT_ERROR"))

## Lesson 9 — Fragments for HiGNN and JT-VAE intuition

Fragmentation turns a molecule into chemically meaningful pieces. In this project:

- HiGNN uses an atom graph plus fragment-level hierarchy.
- JT-VAE uses a junction tree of molecular substructures to make generation more chemically valid.

In [ ]:
target = Chem.MolFromSmiles("CC(=O)Nc1ccc(OCCN2CCOCC2)cc1")
frags = sorted(BRICS.BRICSDecompose(target))
frag_mols = [Chem.MolFromSmiles(f) for f in frags]
print("Number of BRICS fragments:", len(frags))
for f in frags:
    print(f)
display(Draw.MolsToGridImage([target] + frag_mols, legends=["parent"] + frags, molsPerRow=3, subImgSize=(260, 180)))

## Lesson 10 — Simple validity, uniqueness, novelty metrics for generated molecules

Generative-model papers often report:

- validity: fraction of generated SMILES that parse and sanitize;
- uniqueness: fraction of valid molecules that are non-duplicates;
- novelty: fraction of valid molecules not in the training set;
- property distribution match: QED, LogP, MW, SA-like score, etc.

Validity is necessary but weak. `CCCCCCCCCCCCCCCC` can be valid and boring. A model can be valid, unique, and useless if it ignores the desired chemical space.

In [ ]:
generated = [
    "CCO", "OCC", "c1ccccc1", "C1CC1", "CO(C)C", "not_a_smiles",
    "CC(=O)O", "CC(=O)O", "C[N+](C)(C)C",
]
train = {"CCO", "c1ccccc1"}

valid_mols = []
valid_smiles = []
for smi in generated:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        can = Chem.MolToSmiles(mol)
        valid_mols.append(mol)
        valid_smiles.append(can)

metrics = {
    "n_generated": len(generated),
    "n_valid": len(valid_smiles),
    "validity": len(valid_smiles) / len(generated),
    "uniqueness": len(set(valid_smiles)) / max(1, len(valid_smiles)),
    "novelty": len([s for s in set(valid_smiles) if s not in train]) / max(1, len(set(valid_smiles))),
}
print(metrics)
display(Draw.MolsToGridImage(valid_mols, legends=valid_smiles, molsPerRow=4, subImgSize=(200, 160)))

## Lesson 11 — Protein sequences for DeepAffinity

RDKit handles compounds, not proteins as sequences. For DeepAffinity-like models, a common starting point is amino-acid tokenization:

- map 20 canonical residues plus unknown/padding to integers;
- embed tokens;
- use CNN/RNN/Transformer/attention pooling;
- combine protein representation with compound representation.

The cheminformatics trap: a compound--protein affinity model has two different representation worlds. Bad preprocessing on either side can dominate performance.

In [ ]:
aa = "ACDEFGHIKLMNPQRSTVWY"
stoi = {ch: i + 1 for i, ch in enumerate(aa)}
stoi["X"] = len(stoi) + 1
PAD = 0

def tokenize_protein(seq, max_len=32):
    arr = np.zeros(max_len, dtype=np.int64)
    for i, ch in enumerate(seq[:max_len]):
        arr[i] = stoi.get(ch, stoi["X"])
    return arr

seq = "MKWVTFISLLFLFSSAYSRGVFRRDAHKSEVA"
tokens = tokenize_protein(seq)
print(tokens)
print("length before truncation:", len(seq), "encoded length:", len(tokens))

## Lesson 12 — A practical preprocessing checklist

Before training molecular ML models:

1. Parse SMILES and log failures.
2. Canonicalize duplicates.
3. Decide what to do with salts/mixtures.
4. Decide protonation/tautomer policy.
5. Preserve stereochemistry if labels depend on it.
6. Compute descriptors for sanity plots.
7. Use scaffold/time/assay-aware splits where appropriate.
8. Save processed inputs to `data/`.
9. Save model weights/checkpoints to `models/`.
10. Record package versions with `Environment_Hardware_Check.ipynb`.

This is the unglamorous layer where many model papers quietly win or lose.

## Lesson 13 — Standardization: salts, charges, fragments, and neutralization

Before modeling, decide what a "molecule" means.

Common preprocessing:

- keep largest organic fragment;
- remove salts/solvents;
- normalize charges;
- canonicalize tautomers when appropriate;
- preserve or discard stereochemistry deliberately;
- reject mixtures if the model expects one molecular graph.

This is not clerical. Standardization changes the model input and sometimes the label meaning.

In [ ]:
from rdkit.Chem.MolStandardize import rdMolStandardize

examples = ["CC(=O)[O-].[Na+]", "Cl.CN1CCN(CC1)c1ncccn1", "CCO.O"]
lfc = rdMolStandardize.LargestFragmentChooser()
uncharger = rdMolStandardize.Uncharger()
rows = []
for smi in examples:
    mol = Chem.MolFromSmiles(smi)
    largest = lfc.choose(mol)
    uncharged = uncharger.uncharge(largest)
    rows.append({
        "input": smi,
        "largest_fragment": Chem.MolToSmiles(largest),
        "uncharged_largest": Chem.MolToSmiles(uncharged),
    })
display(pd.DataFrame(rows))

## Lesson 14 — Substructure search with SMARTS

SMARTS is a query language for molecular patterns. Use it for:

- functional-group filters;
- dataset audits;
- scaffold/series analysis;
- excluding obvious reactive groups;
- interpreting attention hits.

SMARTS can be subtle; test patterns on positive and negative examples before trusting them.

In [ ]:
patterns = {
    "phenyl": "c1ccccc1",
    "carboxylic_acid": "C(=O)[OH]",
    "tertiary_amine": "[NX3;H0;!$(NC=O)]",
}
rows = []
for name, smi in library.items():
    mol = Chem.MolFromSmiles(smi)
    row = {"name": name}
    for label, smarts in patterns.items():
        row[label] = mol.HasSubstructMatch(Chem.MolFromSmarts(smarts))
    rows.append(row)
display(pd.DataFrame(rows))

## Lesson 15 — Highlighting substructures and atom importance

Many interpretability plots eventually become "which atoms/bonds mattered?" RDKit can highlight atoms from a SMARTS match, attention score, gradient attribution, or fragment membership.

In [ ]:
mol = Chem.MolFromSmiles(library["aspirin"])
patt = Chem.MolFromSmarts("C(=O)O")
match = mol.GetSubstructMatch(patt)
print("carboxyl match atom indices:", match)
display(Draw.MolToImage(mol, highlightAtoms=list(match), size=(420, 260)))

## Lesson 16 — Similarity search and nearest-neighbor baselines

Before celebrating a deep model, compare it to a fingerprint nearest-neighbor baseline. If test molecules are close to train molecules, a simple baseline may perform surprisingly well.

For a regression label, a simple baseline is:

1. compute train/test fingerprints;
2. find nearest train molecule by Tanimoto;
3. predict that train molecule's label.

This is not fancy. That is exactly why it is useful.

In [ ]:
toy_labels = np.array([0.1, 0.4, 1.2, 1.0, 2.5])
query = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")  # aspirin, aromatic ester acid
qfp = fpgen.GetFingerprint(query)
sims = np.array([DataStructs.TanimotoSimilarity(qfp, fp) for fp in fps])
best = int(sims.argmax())
print("nearest:", names[best], "similarity:", sims[best], "predicted label:", toy_labels[best])
plt.bar(names, sims)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Tanimoto to query")
plt.title("Nearest-neighbor baseline intuition")
plt.tight_layout()
plt.show()

## Lesson 17 — Molecular graph tensors for generative models

GraphVAE and MolGAN-style models often use padded dense tensors:

- node tensor: `[max_nodes, atom_types]`;
- adjacency/edge tensor: `[bond_types, max_nodes, max_nodes]`;
- mask: which node slots are real.

Pros:

- simple neural decoder output shape;
- easy batching.

Cons:

- fixed maximum graph size;
- many invalid outputs;
- permutation problem: graph node ordering is arbitrary.

In [ ]:
atom_vocab = ["C", "N", "O", "F", "PAD"]
bond_vocab = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.AROMATIC]

def dense_graph_tensors(mol, max_nodes=12):
    atom_to_idx = {a: i for i, a in enumerate(atom_vocab)}
    x = np.zeros((max_nodes, len(atom_vocab)), dtype=np.float32)
    edge = np.zeros((len(bond_vocab), max_nodes, max_nodes), dtype=np.float32)
    mask = np.zeros(max_nodes, dtype=bool)
    for atom in mol.GetAtoms():
        i = atom.GetIdx()
        if i >= max_nodes:
            continue
        sym = atom.GetSymbol()
        x[i, atom_to_idx.get(sym, atom_to_idx["PAD"])] = 1
        mask[i] = True
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        if i < max_nodes and j < max_nodes and bond.GetBondType() in bond_vocab:
            k = bond_vocab.index(bond.GetBondType())
            edge[k, i, j] = edge[k, j, i] = 1
    x[~mask, atom_to_idx["PAD"]] = 1
    return x, edge, mask

x_dense, e_dense, mask = dense_graph_tensors(Chem.MolFromSmiles(library["caffeine"]), max_nodes=16)
print("node tensor:", x_dense.shape, "edge tensor:", e_dense.shape, "mask:", mask.shape, "real nodes:", mask.sum())

## Lesson 18 — Matching generated graphs back to molecules

Decoding graph tensors is harder than encoding:

1. choose atom types;
2. choose bond types;
3. remove padding;
4. build RDKit `RWMol`;
5. sanitize;
6. reject or repair invalid valence/aromaticity.

This is why JT-VAE uses stronger chemical structure. It narrows the decoder's search space to more plausible molecular assemblies.

In [ ]:
def simple_decode_atoms_bonds(atom_symbols, bonds):
    rw = Chem.RWMol()
    for sym in atom_symbols:
        rw.AddAtom(Chem.Atom(sym))
    for i, j, btype in bonds:
        rw.AddBond(i, j, btype)
    mol = rw.GetMol()
    try:
        Chem.SanitizeMol(mol)
        return mol, Chem.MolToSmiles(mol), None
    except Exception as exc:
        return mol, None, repr(exc)

mol_ok, smi_ok, err_ok = simple_decode_atoms_bonds(["C", "C", "O"], [(0, 1, Chem.BondType.SINGLE), (1, 2, Chem.BondType.SINGLE)])
mol_bad, smi_bad, err_bad = simple_decode_atoms_bonds(["C"], [(0, 0, Chem.BondType.SINGLE)])
print("valid decode:", smi_ok, err_ok)
print("invalid decode:", smi_bad, err_bad)

## Lesson 19 — Descriptor scaling and feature hygiene

Classic descriptors live on wildly different scales: molecular weight, LogP, TPSA, counts, binary flags. Neural networks usually prefer standardized continuous inputs.

Rules:

- fit scalers on train only;
- transform validation/test with train statistics;
- save scaler parameters;
- never compute global normalization before splitting.

In [ ]:
cont = df[["MW", "LogP", "TPSA", "HBD", "HBA", "QED"]].copy()
mu = cont.mean()
sigma = cont.std(ddof=0).replace(0, 1)
z = (cont - mu) / sigma
display(z.round(2))
print("means after scaling:")
display(z.mean().round(6))

## Lesson 20 — RDKit failure handling and audit logs

Production cheminformatics code should never silently drop molecules. Make an audit table:

- original ID;
- original SMILES;
- parse status;
- standardized SMILES;
- reason for exclusion;
- descriptor sanity checks.

This makes experiments reproducible and defensible.

In [ ]:
raw = ["CCO", "not_smiles", "C1CC1", "C[N+](C)(C)C", ""]
audit = []
for i, smi in enumerate(raw):
    mol = Chem.MolFromSmiles(smi) if smi else None
    if mol is None:
        audit.append({"row": i, "input": smi, "status": "failed_parse", "canonical": None, "MW": None})
    else:
        audit.append({"row": i, "input": smi, "status": "ok", "canonical": Chem.MolToSmiles(mol), "MW": Descriptors.MolWt(mol)})
audit_df = pd.DataFrame(audit)
display(audit_df)
audit_df.to_csv(DATA_DIR / "rdkit_audit_demo.csv", index=False)
print("Saved:", DATA_DIR / "rdkit_audit_demo.csv")

## Lesson 21 — What to learn next in cheminformatics

After this notebook, good next steps are:

- TeachOpenCADD talktorials on compound datasets, similarity, docking, and machine learning.
- RDKit UGM notebooks and blog posts by Greg Landrum.
- DeepChem examples for molecular datasets and featurizers.
- Basic docking and structure-based design if you want to connect sequence affinity models to 3D pockets.

The biggest conceptual step is accepting that "SMILES → tensor" is not a neutral act. It is a scientific modeling choice wearing a programmer's hoodie.